# Governance Failure Notifier

Called by the pipeline whenever a critical activity fails (the 5 governance notebooks, plus
the 3 owner/admin/deletion email send-loops). Builds one Xebia-branded HTML alert and queues
it to `email_outbox` with `email_type="pipeline_failure_alert"` — the pipeline then sends it
the same way it sends every other email type (Lookup → Filter → ForEach → Office365Email).

**No API calls. No admin permission. Reads and writes Delta tables only.**

One alert per failed activity per run — if a `ForEach` send-loop fails partway through, this
reports the loop as a whole failing, not which specific recipient's send failed. To see who
didn't get sent after such an alert, check `email_outbox` for `status='pending'` rows from
the same `pipeline_run_id`.

## 1. Parameters

In [ ]:
# Set by the pipeline via this activity's Base parameters (or edit these for a manual test run).
activity_name  = "Unknown activity"   # e.g. "Run Inventory Scan"
error_message  = "No error message provided"
pipeline_name  = "Governance_Pipeline"
run_id         = ""
workspace_id   = ""

## 2. Setup

In [ ]:
import pandas as pd
import uuid
import logging
from datetime import datetime, timezone

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("failure_notifier")

notifier_run_id = str(uuid.uuid4())
run_timestamp   = datetime.now(timezone.utc).isoformat()
run_date_str    = datetime.now(timezone.utc).strftime("%B %d, %Y")

log.info(f"Activity that failed: {activity_name}")
log.info(f"Pipeline run ID: {run_id}")

## 3. Load config

In [ ]:
df_cfg = spark.sql("SELECT config_key, config_value FROM governance_config").toPandas()
config = dict(zip(df_cfg["config_key"], df_cfg["config_value"]))

admin_email = config.get("admin_email", "").strip()
if not admin_email:
    admin_email = "<ADMIN_EMAIL>"
    log.warning("CRITICAL: admin_email missing from governance_config — using hardcoded fallback. Set it explicitly.")

failure_notify_email = config.get("pipeline_failure_notify_email", "").strip()
if not failure_notify_email:
    failure_notify_email = admin_email
    log.info(f"pipeline_failure_notify_email not set — falling back to admin_email ({admin_email})")

logo_url = config.get("logo_url", "")
log.info(f"Failure alert recipient: {failure_notify_email}")

## 4. Build the alert email

Same Xebia brand styling as `Governance_Email_Generator.ipynb` — deliberately kept as a
self-contained copy rather than a shared import, since Fabric notebooks can't import each
other, but the color constants and header/footer shape are identical on purpose.

In [ ]:
XEBIA_PURPLE    = "#6a1b6a"
XEBIA_PURPLE_LT = "#f3e8f3"
WHITE           = "#ffffff"
GRAY_BG         = "#f7f7f7"
GRAY_BORDER     = "#e0e0e0"
GRAY_TEXT       = "#666666"
BLACK_TEXT      = "#333333"
RED_ACCENT      = "#d32f2f"
RED_BG          = "#fdecea"

logo_html = (f'<img src="{logo_url}" alt="Xebia" width="130" style="display:block;border:0;" />'
             if logo_url else
             f'<span style="font-size:26px;font-weight:700;color:{XEBIA_PURPLE};letter-spacing:0.5px;">Xebia</span>')

monitoring_hub_url = f"https://app.fabric.microsoft.com/workspaces/{workspace_id}/monitoringhub" if workspace_id else "https://app.fabric.microsoft.com"

subject = f"[ALERT] {pipeline_name} — '{activity_name}' failed"

html = f"""
<table width="100%" cellpadding="0" cellspacing="0" border="0" style="max-width:680px;margin:0 auto;font-family:Segoe UI,Arial,sans-serif;background:{WHITE};">
<tr><td style="height:5px;background:{RED_ACCENT};font-size:0;line-height:0;">&nbsp;</td></tr>
<tr><td style="padding:20px 28px 16px;border-left:1px solid {GRAY_BORDER};border-right:1px solid {GRAY_BORDER};">
    <table width="100%" cellpadding="0" cellspacing="0" border="0">
    <tr>
        <td align="left" valign="middle">{logo_html}</td>
        <td align="right" valign="middle" style="font-size:11px;color:{GRAY_TEXT};letter-spacing:0.5px;text-transform:uppercase;">Fabric Workspace Governance</td>
    </tr>
    </table>
</td></tr>
<tr><td style="padding:0 28px 20px;border-left:1px solid {GRAY_BORDER};border-right:1px solid {GRAY_BORDER};border-bottom:1px solid #f0f0f0;">
    <h1 style="margin:0 0 4px;font-size:20px;font-weight:600;color:{BLACK_TEXT};">Pipeline activity failed</h1>
    <p style="margin:0;font-size:13px;color:{GRAY_TEXT};">{pipeline_name} &bull; {run_date_str}</p>
</td></tr>
<tr><td style="background:{WHITE};padding:16px 28px;border-left:1px solid {GRAY_BORDER};border-right:1px solid {GRAY_BORDER};">
    <table width="100%" cellpadding="0" cellspacing="0" style="border:1px solid {GRAY_BORDER};border-radius:6px;border-collapse:collapse;font-size:13px;">
        <tr><td style="background:{XEBIA_PURPLE_LT};padding:8px 12px;font-weight:600;width:140px;border-bottom:1px solid {GRAY_BORDER};">Activity</td>
            <td style="padding:8px 12px;border-bottom:1px solid {GRAY_BORDER};">{activity_name}</td></tr>
        <tr><td style="background:{XEBIA_PURPLE_LT};padding:8px 12px;font-weight:600;border-bottom:1px solid {GRAY_BORDER};">Pipeline</td>
            <td style="padding:8px 12px;border-bottom:1px solid {GRAY_BORDER};">{pipeline_name}</td></tr>
        <tr><td style="background:{XEBIA_PURPLE_LT};padding:8px 12px;font-weight:600;border-bottom:1px solid {GRAY_BORDER};">Run ID</td>
            <td style="padding:8px 12px;border-bottom:1px solid {GRAY_BORDER};">{run_id}</td></tr>
        <tr><td style="background:{XEBIA_PURPLE_LT};padding:8px 12px;font-weight:600;">Monitoring Hub</td>
            <td style="padding:8px 12px;"><a href="{monitoring_hub_url}">Open Monitoring Hub</a> and search for run ID {run_id}</td></tr>
    </table>
    <p style="margin:16px 0 6px;font-size:13px;font-weight:600;color:{BLACK_TEXT};">Error message</p>
    <pre style="margin:0;padding:12px 14px;background:{RED_BG};border-left:4px solid {RED_ACCENT};border-radius:6px;font-size:12px;color:{BLACK_TEXT};white-space:pre-wrap;word-break:break-word;">{error_message}</pre>
</td></tr>
<tr><td style="background:{GRAY_BG};padding:16px 28px;border-radius:0 0 8px 8px;border:1px solid {GRAY_BORDER};border-top:none;text-align:center;">
    <p style="margin:0;font-size:12px;color:{GRAY_TEXT};">
        Xebia &bull; Fabric Workspace Governance &bull; Automated failure alert
    </p>
</td></tr>
</table>
"""

log.info(f"Alert built: {len(html)} chars")

## 5. Queue to email_outbox

In [ ]:
outbox_row = {
    "email_id": str(uuid.uuid4()),
    "pipeline_run_id": run_id or notifier_run_id,
    "email_type": "pipeline_failure_alert",
    "recipient": failure_notify_email,
    "subject": subject,
    "body_html": html,
    "status": "pending",
    "created_at": run_timestamp,
    "workspace_id": workspace_id,
}

spark.createDataFrame(pd.DataFrame([outbox_row]).astype(str)).write.format("delta").mode("append").saveAsTable("email_outbox")
log.info(f"\u2714 Failure alert queued for {failure_notify_email} (pipeline_run_id={run_id or notifier_run_id})")

# Exit with this row's email_id so the pipeline can pass it straight to the
# Governance_Send_Failure_Alert child pipeline -- filtering delivery on the exact row
# this run wrote, instead of just pipeline_run_id, which could match more than one
# alert if two activities happen to fail in the same run.
notebookutils.notebook.exit(outbox_row["email_id"])